# Stata Analysis Code Template

This is a template for Stata analysis code. It is designed to be used in a Jupyter notebook.

* Key rules: 
    - EVERY cell should be self-contained and executable, so all global settings should be repeated in every cell.
    - Each cell usually write one table, panel or figure.
    - Do not write custom `program define` wrappers, Mata code, or meta/diagnostic machinery (postfiles, model-metadata logging, coefficient dumps); call estimators directly and store results with `est store` (plus `estadd` for table rows).
    - Keep the codes as few lines as possible, minimize unnecessary code beyond shown in the examples below.

* Output rules (always follow):
    - Tables: export to four formats with the same basename — `.md`, `.rtf`, `.csv`, `.tex` — using the `esttab` options shown below (`cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label`; add `r2_p` for nonlinear models).
    - Figures: export to both `.pdf` and `.png`.
    - No extra side files (`.ster`, metadata, diagnostic dumps); the four formats are the deliverable.

* Entry file: one `reg.dta` file per sample (one for a project that uses only one baseline sample)

## Example: baseline models

## Example: cross-sectional tests

## Example: descriptive statistics

In [ ]:
version 18
set more off

capture mkdir "artifacts"

global y "eps_price"
global x "dum ret dum_return"

use "reg.dta", clear
est clear
regress $y $x
quietly count if e(sample) & inrange(fyear, 1951, 2001)
local n_original = r(N)
quietly count if e(sample) & inrange(fyear, 2002, 2006)
local n_added = r(N)
quietly summarize fyear if e(sample) & inrange(fyear, 2002, 2006), meanonly
local last_added = r(max)
display as text "EXT_SAMPLE s1 `n_original' `n_added' `last_added'"
est store m1
estadd scalar n_original = `n_original' : m1
estadd scalar n_added = `n_added' : m1
estadd scalar last_added = `last_added' : m1

esttab m1 using "tables/annual_pooled_eq1.md", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label
esttab m1 using "tables/annual_pooled_eq1.rtf", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label
esttab m1 using "tables/annual_pooled_eq1.csv", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label
esttab m1 using "tables/annual_pooled_eq1.tex", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label
esttab m1 using "artifacts/annual_pooled_eq1.csv", cells(b(star fmt(6)) p(par fmt(4))) starlevels(* 0.1 ** 0.05 *** 0.01) stats(N n_original n_added last_added, fmt(0 0 0 0) labels("Total estimation observations" "Original-period firm-years" "Added-period firm-years" "Last added fiscal year")) replace label
est clear


In [ ]:
version 18
set more off

global descvars "eps_price dum ret dum_return ret_days"

use "reg.dta", clear
est clear
estpost tabstat $descvars, stat(count mean sd p25 p50 p75) columns(statistics)
esttab using "tables/annual_descriptives.md", replace label cells("count(fmt(0)) mean(fmt(4)) sd(fmt(4)) p25(fmt(4)) p50(fmt(4)) p75(fmt(4))")
est clear
estpost tabstat $descvars, stat(count mean sd p25 p50 p75) columns(statistics)
esttab using "tables/annual_descriptives.rtf", replace label cells("count(fmt(0)) mean(fmt(4)) sd(fmt(4)) p25(fmt(4)) p50(fmt(4)) p75(fmt(4))")
est clear
estpost tabstat $descvars, stat(count mean sd p25 p50 p75) columns(statistics)
esttab using "tables/annual_descriptives.csv", replace label cells("count(fmt(0)) mean(fmt(4)) sd(fmt(4)) p25(fmt(4)) p50(fmt(4)) p75(fmt(4))")
est clear
estpost tabstat $descvars, stat(count mean sd p25 p50 p75) columns(statistics)
esttab using "tables/annual_descriptives.tex", replace label cells("count(fmt(0)) mean(fmt(4)) sd(fmt(4)) p25(fmt(4)) p50(fmt(4)) p75(fmt(4))")
est clear
